# 🧠 Mini-Batch Gradient Descent and Vectorization

Welcome to the hands-on explanation notebook for **Mini-Batch Gradient Descent**! In this notebook, we will:
1. Compare Batch GD, Stochastic GD (SGD), and Mini-Batch GD in detail.
2. Implement **vectorized Mini-Batch GD** from scratch using NumPy.
3. Compare the optimization trajectories of all three methods on a 2D loss contour map.
4. Visualize the loss reduction curves over update steps.
5. Explain GPU matrix multiplication (vectorization) and tensor shapes.
6. Connect these parameters to YOLO's `batch` argument and CUDA Out of Memory (OOM) errors.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. Simulating Data for Linear Regression

We generate 100 samples following the linear process $y = 3x + 1 + \text{noise}$.

In [ ]:
n_samples = 100
X = np.random.rand(n_samples, 1)
y = 3 * X + 1 + np.random.normal(0, 0.15, (n_samples, 1))

## 2. Vectorized Mini-Batch GD from Scratch

Let's implement Mini-Batch GD.
Steps per epoch:
1. Shuffle indices randomly.
2. Slices of size $B$ (batch size) are extracted.
3. Perform forward pass and vectorized gradient computation on the mini-batch:
   $$\text{dw} = \frac{1}{B} \mathbf{X}_{\text{batch}}^T (\hat{\mathbf{y}} - \mathbf{y}_{\text{batch}})$$
   $$\text{db} = \frac{1}{B} \sum (\hat{\mathbf{y}} - \mathbf{y}_{\text{batch}})$$
4. Update weights immediately.

In [ ]:
def run_mini_batch_gd(X, y, batch_size=10, lr=0.1, epochs=10):
    w, b = 0.0, 0.0
    m = len(X)
    history = [[w, b]]
    
    for _ in range(epochs):
        indices = np.random.permutation(m)
        X_shuffled = X[indices]
        y_shuffled = y[indices]
        
        for i in range(0, m, batch_size):
            X_batch = X_shuffled[i : i + batch_size]
            y_batch = y_shuffled[i : i + batch_size]
            B = len(X_batch)
            
            preds = w * X_batch + b
            error = preds - y_batch
            
            # Vectorized gradients
            dw = (1 / B) * np.dot(X_batch.T, error)[0, 0]
            db = (1 / B) * np.sum(error)
            
            w -= lr * dw
            b -= lr * db
            history.append([float(w), float(b)])
            
    return np.array(history)

# Run optimization for all three modes
path_bgd = run_mini_batch_gd(X, y, batch_size=100, lr=0.1, epochs=50)
path_sgd = run_mini_batch_gd(X, y, batch_size=1, lr=0.01, epochs=2)
path_mbgd = run_mini_batch_gd(X, y, batch_size=10, lr=0.05, epochs=5)

## 3. Visualizing Trajectories on the contour map

Let's plot the paths of all three models on the same 2D Loss Landscape contour map.

In [ ]:
# Compute loss grid
w_vals = np.linspace(-0.5, 4.5, 100)
b_vals = np.linspace(-0.5, 2.5, 100)
W, B = np.meshgrid(w_vals, b_vals)

Z = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        w_tmp, b_tmp = W[i, j], B[i, j]
        Z[i, j] = (1 / (2 * n_samples)) * np.sum((w_tmp * X + b_tmp - y) ** 2)

# Plot paths
plt.figure(figsize=(10, 8))
contours = plt.contour(W, B, Z, levels=25, cmap='viridis')
plt.clabel(contours, inline=1, fontsize=8)

plt.plot(path_bgd[:, 0], path_bgd[:, 1], color='red', marker='o', linewidth=2.5, label='Batch GD (Size = 100)')
plt.plot(path_sgd[:, 0], path_sgd[:, 1], color='orange', alpha=0.6, linewidth=1.5, label='Stochastic GD (Size = 1)')
plt.plot(path_mbgd[:, 0], path_mbgd[:, 1], color='cyan', marker='x', linewidth=2, label='Mini-Batch GD (Size = 10)')

plt.scatter(3.0, 1.0, color='blue', s=120, marker='*', zorder=5, label='Target Minimum')
plt.xlabel('Weight (w)')
plt.ylabel('Bias (b)')
plt.title('Contour Map Comparison of Gradient Descent Variants')
plt.legend()
plt.show()

Observe:
-   **Batch GD:** Extremely smooth, but performs only 1 update per epoch.
-   **Stochastic GD:** Very noisy, wiggling erratically from side to side.
-   **Mini-Batch GD:** The optimal middle ground: it follows a stable, slightly wavy trajectory but makes frequent updates, converging very quickly and efficiently.

## 💡 Connection to YOLO and Deep Learning
*   **The `batch` parameter:** When training YOLO models, the batch size is set using the `batch` argument:
    ```bash
    yolo train model=yolo11n.pt data=data.yaml batch=16
    ```
*   **CUDA Out of Memory (OOM) Errors:** If you choose a batch size that is too large (e.g. `batch=64` or `128` on a small GPU), the tensor of size `[64, 3, 640, 640]` containing the images and all their intermediate convolutional activation maps will exceed the GPU's memory limit, triggering a CUDA OOM crash.
*   **Fix:** If you see an OOM error, reduce the batch size (e.g., `batch=16` or `batch=8`).